# 02 · Store

*Part 1, section 2.4 and Part 2, section 2.* Traces are the raw signal; the **evaluation dataset**
is the curated one, and it doubles as the regression suite. It grows from two places: production
failures that a person labeled with the right answer, and hand-written **golden cases** that cover
behavior that has not failed yet.

Two properties make it trustworthy later:

* **provenance**: every row says where it came from (`source`, `source_trace`, `author`, `theme`)
* **a holdout**: a slice no optimizer, analyst or approver ever sees, only the gate

In [1]:
import _nb as nb

experiment_id = nb.setup()

/Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


env file      /Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/harness/.env
tracking      sqlite:////Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/harness/.selfimprove/notebooks/mlflow.db   experiment walkthrough
agent         python: examples.support_agent:answer
instructions  prompts:/walkthrough-instructions  (seed /Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/harness/examples/instructions.md)
judge         anthropic:/claude-opus-5-5  (credentials found)
first pass    typesafe:/jev-latest  (re-judge p(pass) in 0.50-0.80)
improver      analyst   auto-promote False
mode          live


In [2]:
import mlflow
from mlflow.genai.datasets import create_dataset
from selfimprove import config, store, traces

dataset = create_dataset(name=config.settings.dataset_name, experiment_id=[experiment_id],
                         tags={"purpose": "regression"})
dataset.name

'walkthrough-regressions'

**Under the hood:** the harness's `store.dataset()` fetches the dataset by name and creates it on
first use, with the tags `purpose` and `holdout_fraction`. This cell made it with `purpose` only;
`approve` adds the rest in notebook 04.

**MLflow in this step**

* `create_dataset(name=..., experiment_id=[...], tags=...)` (from `mlflow.genai.datasets`): an
  evaluation dataset, an entity in the tracking store with an id (`d-...`), linked to the experiment,
  with tags
* `get_dataset(name=...)`: fetches it again on every later call

In [3]:
from mlflow.genai.datasets import get_dataset

ds = get_dataset(name=config.settings.dataset_name)
print(ds.dataset_id, "| experiments", ds.experiment_ids)
print("tags", {k: v for k, v in ds.tags.items() if not k.startswith("mlflow.")})

d-7c193ec4524a4febaa3117e245e0f1ff | experiments ['1']
tags {'purpose': 'regression'}


## 1. Failures with an expectation become records

Only **human** failures that carry an expectation qualify. A judge verdict never becomes an
expectation: a person or a golden case is the source of truth for what correct means.

In [4]:
labeled_fails = [
    t for t in traces.production(f"{traces.FAILED} AND {traces.HAS_EXPECTATION}")
    if (fb := traces.human_feedback(t)) is not None and traces.is_fail(fb.value)
]
print(len(labeled_fails), "human-labeled failures with an expected response")

rec = store.record_from_trace(labeled_fails[0], source="human_fail", author="you")
rec

17 human-labeled failures with an expected response


{'inputs': {'question': 'I lost my API key. Can you show it to me again?'},
 'expectations': {'expected_response': 'No. Keys are shown only once. Create a new one at Settings → Developer → API keys.'},
 'tags': {'source': 'human_fail',
  'author': 'you',
  'split': 'train',
  'added_at': '1793217170638',
  'source_trace': 'tr-48fa28afe37e3abb8dec9b5e0d9886fa',
  'prompt_version': '1'}}

**Under the hood:** `store.record_from_trace()` writes nothing. It returns a plain dict in the shape
`merge_records()` takes: `inputs`, `expectations`, and `tags` for provenance and the split.

**MLflow in this step**

* `trace.data.request`: the trace's input
* `trace.info.assessments`: its expectations (the `expected_response` a reviewer logged)
* record tags: MLflow stores the provenance and split tags on each dataset record

The split is chosen by a **hash of the input**, so a case lands in the same split forever, however
many times it fails:

In [5]:
for q in ["How long can I wait to request a refund?", "Can I pay in Canadian dollars?",
          "How many teammates can I invite on Pro?"]:
    print(f"{store.split_for(q):<8} {q}")

holdout  How long can I wait to request a refund?
train    Can I pay in Canadian dollars?
train    How many teammates can I invite on Pro?


## 2. Forty complaints, one test

The same question failing many times is one bug. Deduplicate before merging, or the suite bloats
instead of improving (Part 2, section 5).

In [6]:
records, seen = [], []
for t in labeled_fails:
    r = store.record_from_trace(t, source="human_fail", author="you")
    if store.near_duplicate(store.question_of(r), seen) is None:
        records.append(r)
        seen.append(r)
print(f"{len(labeled_fails)} failures -> {len(records)} distinct tests")
dataset.merge_records(records)

17 failures -> 9 distinct tests


**Under the hood:** the cell deduplicates first because MLflow only recognizes exact repeats. Merging
the same failure twice does not grow the suite, but two different phrasings of one question would.

**MLflow in this step**

* `dataset.merge_records(records)`: an upsert. MLflow hashes each record's `inputs`; a record whose
  inputs already exist is merged into the existing row (its expectations and tags are updated)
  instead of being added again

## 3. Golden cases

Written by a person, including the **holdout** cases (`holdout: true`). These are the cases that
tell you whether an improvement generalizes or just memorized the failures.

In [7]:
print(open("examples/golden.yaml").read()[:900])

# Golden cases: written by a person, merged into the dataset by `selfimprove init`.
# holdout: true  -> only the gate ever sees it (no improver, analyst or approver does)
# holdout: false -> part of the CI suite the optimizer trains on
# key_facts are regexes for the `key_facts` reference scorer.
- id: currencies
  holdout: true
  inputs:
    question: Can I pay in Canadian dollars?
  expectations:
    expected_response: No. Acme Cloud bills in USD, EUR, and GBP only.
    key_facts:
    - USD
    - EUR
    - GBP
- id: refund_where
  holdout: true
  inputs:
    question: Bought the monthly plan yesterday by mistake. Refund?
  expectations:
    expected_response: Yes, you are within the 14-day window. Request it at Settings → Billing → Request refund.
    key_facts:
    - 14[- ]day
- id: sso_enterprise
  holdout: true
  inputs:
    question: Which plan do I need for SAML single sign-on?
  


In [8]:
print("+", store.seed_golden(), "golden cases")
print(store.summary())

+ 6 golden cases
15 records (7 train, 8 holdout) by source {'human_fail': 9, 'golden': 6}


**Under the hood:** `store.seed_golden()` reads `golden.yaml`, builds one record per case with
`source=golden`, `author=human` and the split from `holdout:`, and skips inputs that are already in
the dataset.

**MLflow in this step**

* `dataset.merge_records(records)`: adds the golden cases
* `get_dataset(...).to_df()`: how `store.summary()` reads the rows back

Each row is a dataset record with its own id; the split and source live in the record's tags:

In [9]:
import pandas as pd

df = get_dataset(name=config.settings.dataset_name).to_df()
print(len(df), "records; columns:", list(df.columns))
pd.DataFrame(list(df["tags"])).groupby(["split", "source"]).size()

15 records; columns: ['inputs', 'outputs', 'expectations', 'tags', 'source_type', 'source_id', 'source', 'created_time', 'dataset_record_id']


split    source    
holdout  golden        6
         human_fail    2
train    human_fail    7
dtype: int64

In [10]:
df = dataset.to_df()
df.assign(question=df["inputs"].map(lambda i: i["question"][:50]),
          split=df["tags"].map(lambda t: t.get("split")),
          source=df["tags"].map(lambda t: t.get("source")))[["question", "split", "source"]]

,question,split,source
0,How do I export all my data?,train,human_fail
1,I bought Pro three weeks ago and barely used i...,train,human_fail
2,How long is a password reset link valid?,train,human_fail
3,What happens if my card payment fails?,train,human_fail
4,I reset my password but still can't log in. Wh...,holdout,human_fail
5,I lost my API key. Can you show it to me again?,train,human_fail
6,Does Acme Cloud integrate with Jira?,train,human_fail
7,I'm on the annual plan. Do I get a longer refu...,train,human_fail
8,How long can I wait to request a refund?,holdout,human_fail
9,Can I use Acme Cloud offline in Chrome?,holdout,golden


Treat the dataset as code: every row carries its provenance (`source`, `author`, `source_trace`),
and every evaluation run logs a digest of the rows it evaluated, so a pass or fail can be traced to
a specific set of cases.

**Next:** [03 · Update and gate](03_update_and_gate.ipynb): the first manual pass of the loop.